In [2]:
import pyvista as pv
import numpy as np

from pyvista.trame.jupyter import launch_server
await launch_server().ready
pv.set_jupyter_backend('trame') 

In [3]:
stl_finger= 'clara_stl_files/012_LHC_WVM_6L2-cube.stl'
surf_finger= pv.read(stl_finger)

xmin, xmax, ymin, ymax, zmin, zmax = surf_finger.bounds
Lx, Ly, Lz = (xmax-xmin), (ymax-ymin), (zmax-zmin)
stl_tol=1e-3
Nx, Ny, Nz = 200,200,300
x = np.linspace(xmin, xmax, Nx)
y = np.linspace(ymin, ymax, Ny)
z = np.linspace(zmin, zmax, Nz)
spacing=[x[1]-x[0],y[1]-y[0],z[1]-z[0]]

stl_tol=1e-3
stl_tolerance=np.min(spacing)*stl_tol

key='Fingers'
stl_materials = {'Fingers': [1e4, 1., 1e4]}
stl_names = {'Fingers': stl_finger}
stl_solids = {m: f'{stl_names[m]}' for m in stl_materials}



In [4]:
def _mark_cells_in_surface(grid, key, stl_materials,stl_solids):
    # Modify the STL mask to account only for the surface
    """
    Marks surface cells in the grid for a given STL solid mask.
    Converts a volumetric mask into a boolean surface mask.
    """
    if key not in stl_solids:
        return 
    
    #grid['surface']=np.zeros([len(),len()])
    for key in stl_solids.keys():
        if len(stl_materials[key]) == 3 and stl_materials[key][2] > 1e3:
            # 1:
            #grad = np.array(grid.compute_derivative(scalars=key)['gradient']) #bool STL mask
            #grad = np.sqrt(grad[:, 0]**2 + grad[:, 1]**2 + grad[:, 2]**2) #from 0 to 255
            #grid[key] = grad.astype(bool)
            
            # 2: optimized but bulky output /voxel surface is several layers thick
            #grad = grid.compute_derivative(scalars=key,gradient=True,preference='cell')['gradient']
            #surface_mask = np.any(np.array(grad) != 0, axis=1)
            
            # 3: slighly less efficient than 2, but more than 1, applying small threshold to accomodate bulkiness
            grad = grid.compute_derivative(scalars=key,gradient=True,preference='cell')['gradient']
            grad_mag = np.linalg.norm(grad, axis=1)
            surface_mask = grad_mag > 1e-3  # tiny threshold to remove noise
            
            grid['surface'] = surface_mask
    
    return surface_mask
            


In [5]:
import numpy as np
from scipy.ndimage import binary_erosion

def _mark_cells_in_surface_optimized(grid, stl_materials, stl_solids):
    """
    Uses binary erosion to find the surface shell. 
    Memory footprint: ~1 bit per cell + 1 bool per cell.
    """
    # 1. Get the dimensions of your rectilinear grid
    # PyVista rectilinear grids have dimensions (nx, ny, nz)
    shape = grid.dimensions - np.array([1, 1, 1]) 

    for key, mask_data in stl_solids.items():
        # Check conditions
        if len(stl_materials.get(key, [])) == 3 and stl_materials[key][2] > 1e3:
            
            # Extract the volumetric mask as a 3D numpy array
            # .point_data or .cell_data depending on your voxelization
            vol_mask = grid.cell_data[key].reshape(shape, order='F')
            
            # Create a shell by subtracting the eroded volume from the original
            # This is mathematically equivalent to the 'gradient' but uses boolean logic
            eroded = binary_erosion(vol_mask)
            surface_3d = vol_mask ^ eroded  # XOR operation finds the difference (the shell)
            
            # Flatten back and store
            grid.cell_data['surface'] = surface_3d.ravel(order='F')
            
    return grid.cell_data.get('surface')

Implicit distance

In [ ]:
grid = pv.RectilinearGrid(x, y, z)
grid.compute_implicit_distance(surf_finger, inplace=True)

grid[key] = grid.point_data_to_cell_data()['implicit_distance'] <= 0.5
inside_voxels = grid.threshold(0.5, scalars="Fingers") 

surface_cell_mask=_mark_cells_in_surface(grid,'Fingers',stl_materials,stl_solids)
surface_voxels=grid.threshold(stl_tolerance, scalars="surface")

pl = pv.Plotter()
pl.add_mesh(surf_finger, color='cyan', opacity=0.1, style='wireframe', label='Original STL')
pl.add_mesh(surface_voxels, color='red', show_edges=True, label='Grid Interior')
pl.add_legend()
pl.show()

In [ ]:
grid[key] = grid.point_data_to_cell_data()['implicit_distance'] <= stl_tolerance
inside_voxels = grid.threshold(0.5, scalars="Fingers") 

surface_cell_mask=_mark_cells_in_surface(grid,'Fingers',stl_materials,stl_solids)
surface_voxels=grid.threshold(0.5, scalars="surface")

pl = pv.Plotter()
pl.add_mesh(surf_finger, color='cyan', opacity=0.1, style='wireframe', label='Original STL')
pl.add_mesh(surface_voxels, color='red', show_edges=True, label='Grid Interior')
pl.add_legend()
pl.show()

Voxelize_rectilinear

In [6]:
stl_finger= 'clara_stl_files/012_LHC_WVM_6L2-cube.stl'
surf_finger= pv.read(stl_finger)
surf_finger=surf_finger.subdivide(1)
vox = surf_finger.voxelize_rectilinear(spacing=0.25)
key = vox.array_names[0] # Usually 'mask'
vox.cell_data['Fingers'] = vox.cell_data.pop(key)


In [7]:
'''_mark_cells_in_surface(vox, 'Fingers', stl_materials, stl_solids)'''
_mark_cells_in_surface_optimized(vox, stl_materials, stl_solids)

pyvista_ndarray([0, 0, 0, ..., 0, 0, 0], shape=(85495232,), dtype=uint8)

In [8]:
surface_voxels=vox.threshold(0.5,scalars='surface')

In [9]:
inside_voxels = vox.threshold(0.5, scalars="Fingers")

In [11]:



#surface_voxels = vox.threshold(1e-3, scalars="surface")


#surface_voxels=vox.threshold(0.5,scalars='surface')

pl = pv.Plotter()
pl.add_mesh(surf_finger, color='cyan', opacity=0.1, style='wireframe', label='Original STL')
pl.add_mesh(inside_voxels, color='red', show_edges=True)
#pl.add_mesh(surface_voxels, color='red', show_edges=True, label='Grid Surface')
pl.add_legend()
pl.show()

ValueError: Empty meshes cannot be plotted. Input mesh has zero points. To allow plotting empty meshes, set `pv.global_theme.allow_empty_mesh = True`